# Surrogate Modelling for Mechanical Simulation Engineers

This notebook is a compact exercise built from the much broader `SMT_Tutorial.ipynb`.

The goal is not to review every surrogate model in SMT. The goal is to show a practical workflow that engineers can reuse when a simulation is expensive:

1. define the design variables and outputs,
2. run a small Design of Experiments (DOE),
3. train a surrogate model,
4. check whether it is accurate enough,
5. use it to screen many more concepts quickly.


## Why this matters

In mechanical simulation work, a single FE run can take minutes or hours. That usually limits how many design variants can be explored.

A surrogate model is useful when:
- each simulation is expensive,
- the input space is low or medium dimensional,
- an approximate answer is acceptable for screening,
- only a smaller set of promising concepts will later be validated with the full solver.

Typical use cases:
- thickness studies,
- material parameter sweeps,
- bead or geometry parameter tuning,
- fast trade-off studies before a detailed optimization loop.


## Learning objectives

After this exercise, the team should be able to:
- generate a DOE with Latin Hypercube Sampling,
- train a few basic surrogate models in SMT,
- evaluate model quality with simple metrics,
- use the best model for fast what-if studies.

This example uses a synthetic response that mimics an expensive mechanical simulation. Replace it later with your own solver results.


## What is a surrogate model?

A surrogate model is a fast mathematical approximation of an expensive simulation.

In this context:
- the expensive model is a structural simulation,
- the inputs are design parameters such as thickness, material strength, and bead radius,
- the output is `intrusion`,
- the surrogate is trained on a limited number of simulated points,
- then it is used to estimate intrusion for new parameter combinations that have not yet been simulated.

This is useful for screening, design-space exploration, and pre-optimization.


## SMT techniques used here

We will compare three surrogate techniques from SMT.

### 1. `LS` : Linear regression
- Assumes the response changes linearly with the inputs.
- Very fast and easy to understand.
- Good as a baseline.
- Usually too simple when the simulation response is curved or has interactions.

### 2. `QP` : Quadratic polynomial
- Adds curvature and interaction terms.
- Still simple and interpretable.
- Often works well when the response is smooth and not too complex.
- Can struggle if the true response shape is more irregular.

### 3. `KRG` : Kriging
- Builds a flexible interpolation model from the training data.
- Usually stronger than simple polynomials for nonlinear engineering responses.
- Works well when the number of simulations is limited and the response is smooth but not purely polynomial.
- Often a strong default choice for surrogate modelling in engineering.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, r2_score

from smt.sampling_methods import LHS
from smt.surrogate_models import LS, QP, KRG
from smt.utils.misc import compute_q2, compute_rmse


## Case study

We will mimic a structural simulation with three design variables:
- `t` : sheet thickness in mm,
- `sigma_y` : yield strength in MPa,
- `r` : bead radius in mm.

The output is a single performance measure called `intrusion` in mm.
Smaller is better.

The function below is only a stand-in for a real solver. It is nonlinear enough to show why a surrogate can outperform simple regression.


In [ ]:
# Design variable bounds: [min, max]
xlimits = np.array([
    [0.8, 2.0],    # thickness t [mm]
    [180.0, 420.0],# yield strength sigma_y [MPa]
    [3.0, 12.0],   # bead radius r [mm]
])


def mechanical_response(x):
    """Synthetic stand-in for an expensive mechanical simulation."""
    x = np.atleast_2d(x)
    t = x[:, 0]
    sigma_y = x[:, 1]
    r = x[:, 2]

    intrusion = (
        27.0
        - 8.5 * t
        - 0.018 * sigma_y
        + 0.55 * r
        + 3.0 * (t - 1.35) ** 2
        + 0.000035 * (sigma_y - 300.0) ** 2
        + 0.09 * (r - 7.0) ** 2
        - 0.010 * t * (sigma_y - 300.0)
        - 0.12 * t * r
        + 0.0025 * r * (sigma_y - 300.0)
        + 0.7 * np.sin(0.9 * r)
    )
    return intrusion.reshape(-1, 1)


var_names = ["thickness t [mm]", "yield strength sigma_y [MPa]", "bead radius r [mm]"]
pair_indices = [(0, 1), (0, 2), (1, 2)]
x_nominal = xlimits.mean(axis=1)


def make_pair_grid(ix, iy, fixed=None, n=45):
    fixed = x_nominal.copy() if fixed is None else np.array(fixed, dtype=float).copy()
    xi = np.linspace(xlimits[ix, 0], xlimits[ix, 1], n)
    yi = np.linspace(xlimits[iy, 0], xlimits[iy, 1], n)
    X, Y = np.meshgrid(xi, yi)

    pts = np.tile(fixed, (n * n, 1))
    pts[:, ix] = X.ravel()
    pts[:, iy] = Y.ravel()
    return X, Y, pts


def normalize01(x):
    return (x - xlimits[:, 0]) / (xlimits[:, 1] - xlimits[:, 0])


## Step 1: Create training and validation data

In a real project:
- the training set would come from expensive solver runs,
- the validation set would be a separate set of solver runs not used during training.

Here we generate both with Latin Hypercube Sampling (LHS). LHS is used because it spreads a limited number of samples efficiently across the full design space, giving better
  coverage than purely random sampling.


In [ ]:
ndoe = 30
nval = 150

sampling_train = LHS(xlimits=xlimits, criterion="ese", seed=7)
sampling_val = LHS(xlimits=xlimits, criterion="ese", seed=21)

xt = sampling_train(ndoe)
yt = mechanical_response(xt)

xval = sampling_val(nval)
yval = mechanical_response(xval)

print(f"Training samples: {len(xt)}")
print(f"Validation samples: {len(xval)}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(xt[:, 0], xt[:, 1], c=yt[:, 0], cmap="viridis", s=70)
axes[0].set_xlabel("thickness t [mm]")
axes[0].set_ylabel("yield strength sigma_y [MPa]")
axes[0].set_title("DOE colored by intrusion")

axes[1].scatter(xt[:, 0], xt[:, 2], c=yt[:, 0], cmap="viridis", s=70)
axes[1].set_xlabel("thickness t [mm]")
axes[1].set_ylabel("bead radius r [mm]")
axes[1].set_title("Second view of DOE")

plt.tight_layout()
plt.show()


### Visualize pairwise response surfaces

Because there are three inputs, the full response cannot be shown in one figure.
A practical approach is to plot pairwise surfaces while keeping the third variable fixed at a nominal value.

This is often how engineers inspect sensitivity and interaction effects before training a surrogate.


In [ ]:
fig = plt.figure(figsize=(18, 5))

for plot_id, (ix, iy) in enumerate(pair_indices, start=1):
    X, Y, pts = make_pair_grid(ix, iy, n=45)
    Z = mechanical_response(pts).reshape(X.shape)

    ax = fig.add_subplot(1, 3, plot_id, projection="3d")
    surf = ax.plot_surface(X, Y, Z, cmap="viridis", linewidth=0, antialiased=True, alpha=0.9)
    ax.scatter(xt[:, ix], xt[:, iy], yt[:, 0], color="black", s=18, alpha=0.55)
    ax.set_xlabel(var_names[ix])
    ax.set_ylabel(var_names[iy])
    ax.set_zlabel("intrusion [mm]")
    fixed_idx = [k for k in range(3) if k not in (ix, iy)][0]
    ax.set_title(f"{var_names[ix]} vs {var_names[iy]}\n{var_names[fixed_idx]} = {x_nominal[fixed_idx]:.2f}")

fig.colorbar(surf, ax=fig.axes, shrink=0.75, pad=0.08, label="intrusion [mm]")
plt.tight_layout()
plt.show()


### Contour maps for the same slices

Contour plots are often easier to read in design reviews than 3D surfaces.
They make it easier to spot low-response regions, gradients, and variable interactions.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (ix, iy) in zip(axes, pair_indices):
    X, Y, pts = make_pair_grid(ix, iy, n=60)
    Z = mechanical_response(pts).reshape(X.shape)

    contour = ax.contourf(X, Y, Z, levels=18, cmap="viridis")
    ax.contour(X, Y, Z, levels=10, colors="white", linewidths=0.5, alpha=0.7)
    ax.scatter(xt[:, ix], xt[:, iy], c="black", s=16, alpha=0.65)
    ax.set_xlabel(var_names[ix])
    ax.set_ylabel(var_names[iy])
    fixed_idx = [k for k in range(3) if k not in (ix, iy)][0]
    ax.set_title(f"{var_names[ix]} vs {var_names[iy]}\n{var_names[fixed_idx]} = {x_nominal[fixed_idx]:.2f}")

fig.colorbar(contour, ax=axes, shrink=0.9, pad=0.02, label="intrusion [mm]")
plt.tight_layout()
plt.show()


### Good DOE versus poor DOE

A surrogate does not only depend on the model type. It also depends strongly on where simulations are placed.
The comparison below contrasts the Latin Hypercube DOE with a deliberately poor clustered DOE.


In [ ]:
rng = np.random.default_rng(12)
center = x_nominal
span = 0.18 * (xlimits[:, 1] - xlimits[:, 0])
x_bad = center + rng.uniform(-1.0, 1.0, size=(ndoe, 3)) * span
x_bad = np.clip(x_bad, xlimits[:, 0], xlimits[:, 1])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

xt_norm = normalize01(xt)
x_bad_norm = normalize01(x_bad)

axes[0].scatter(xt_norm[:, 0], xt_norm[:, 1], c=yt[:, 0], cmap="viridis", s=55)
axes[0].set_title("Good DOE: LHS coverage")
axes[0].set_xlabel("normalized thickness")
axes[0].set_ylabel("normalized yield strength")
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)

axes[1].scatter(x_bad_norm[:, 0], x_bad_norm[:, 1], c="tab:red", s=55)
axes[1].set_title("Poor DOE: clustered points")
axes[1].set_xlabel("normalized thickness")
axes[1].set_ylabel("normalized yield strength")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()


## Step 2: Train a few candidate surrogate models

For a first engineering workflow, three models are enough:
- `LS` : very fast baseline, but only linear,
- `QP` : quadratic polynomial, useful when trends are smooth,
- `KRG` : Kriging, often a strong default for nonlinear responses and small datasets.


In [ ]:
models = {
    "LS": LS(print_prediction=False), # Linear Regression surrogate
    "QP": QP(print_prediction=False), # Quadratic Polynomial surrogate
    "KRG": KRG(theta0=[1e-2] * xt.shape[1], print_prediction=False), # Kriging surrogate
}

results = []
trained_models = {}

for name, model in models.items():
    model.set_training_values(xt, yt)
    model.train()

    yhat_train = model.predict_values(xt)
    yhat_val = model.predict_values(xval)

    results.append({
        "model": name,
        "train_rmse": float(np.sqrt(mean_squared_error(yt, yhat_train))),
        "val_rmse": float(np.sqrt(mean_squared_error(yval, yhat_val))),
        "train_r2": float(r2_score(yt, yhat_train)),
        "val_r2": float(r2_score(yval, yhat_val)),
        "q2": float(compute_q2(model, xval, yval)),
    })
    trained_models[name] = model

results = sorted(results, key=lambda d: d["val_rmse"])
for row in results:
    print(row)


In [ ]:
best_name = results[0]["model"]
best_model = trained_models[best_name]

print(f"Best model on validation RMSE: {best_name}")
print(f"Validation RMSE: {results[0]['val_rmse']:.3f}")
print(f"Validation Q2:   {results[0]['q2']:.3f}")

best_pred = best_model.predict_values(xval)

plt.figure(figsize=(6, 6))
plt.plot(yval, yval, "k-", label="ideal")
plt.scatter(yval, best_pred, color="tab:red", alpha=0.7, label=best_name)
plt.xlabel("True intrusion [mm]")
plt.ylabel("Predicted intrusion [mm]")
plt.title(f"Validation parity plot: {best_name}")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### Compare true and surrogate surfaces

A low global error metric is useful, but engineers also need to see where the surrogate matches the trend and where it drifts.
The plots below compare the true response and the selected surrogate on two important pairwise slices.


In [ ]:
comparison_pairs = [(0, 1), (0, 2)]
fig = plt.figure(figsize=(14, 10))

for row_id, (ix, iy) in enumerate(comparison_pairs):
    X, Y, pts = make_pair_grid(ix, iy, n=45)
    Z_true = mechanical_response(pts).reshape(X.shape)
    Z_pred = best_model.predict_values(pts).reshape(X.shape)

    ax_true = fig.add_subplot(2, 2, 2 * row_id + 1, projection="3d")
    ax_true.plot_surface(X, Y, Z_true, cmap="viridis", linewidth=0, antialiased=True, alpha=0.95)
    ax_true.set_xlabel(var_names[ix])
    ax_true.set_ylabel(var_names[iy])
    ax_true.set_zlabel("intrusion [mm]")
    ax_true.set_title(f"True response: {var_names[ix]} vs {var_names[iy]}")

    ax_pred = fig.add_subplot(2, 2, 2 * row_id + 2, projection="3d")
    ax_pred.plot_surface(X, Y, Z_pred, cmap="plasma", linewidth=0, antialiased=True, alpha=0.95)
    ax_pred.set_xlabel(var_names[ix])
    ax_pred.set_ylabel(var_names[iy])
    ax_pred.set_zlabel("intrusion [mm]")
    ax_pred.set_title(f"{best_name} prediction: {var_names[ix]} vs {var_names[iy]}")

plt.tight_layout()
plt.show()


### Visualize where the surrogate is accurate

The next plot compares contour maps of the true function, the surrogate prediction, and the local error.
That makes it much easier to discuss where the model is reliable and where additional simulations may be needed.


In [ ]:
ix, iy = 0, 1
X, Y, pts = make_pair_grid(ix, iy, n=70)
Z_true = mechanical_response(pts).reshape(X.shape)
Z_pred = best_model.predict_values(pts).reshape(X.shape)
Z_err = np.abs(Z_true - Z_pred)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

c0 = axes[0].contourf(X, Y, Z_true, levels=20, cmap="viridis")
axes[0].scatter(xt[:, ix], xt[:, iy], c="black", s=12, alpha=0.6)
axes[0].set_title("True response")
axes[0].set_xlabel(var_names[ix])
axes[0].set_ylabel(var_names[iy])

c1 = axes[1].contourf(X, Y, Z_pred, levels=20, cmap="plasma")
axes[1].scatter(xt[:, ix], xt[:, iy], c="black", s=12, alpha=0.6)
axes[1].set_title(f"{best_name} prediction")
axes[1].set_xlabel(var_names[ix])
axes[1].set_ylabel(var_names[iy])

c2 = axes[2].contourf(X, Y, Z_err, levels=20, cmap="magma")
axes[2].scatter(xt[:, ix], xt[:, iy], c="white", s=12, alpha=0.7)
axes[2].set_title("Absolute error")
axes[2].set_xlabel(var_names[ix])
axes[2].set_ylabel(var_names[iy])

fig.colorbar(c0, ax=axes[0], shrink=0.85, label="intrusion [mm]")
fig.colorbar(c1, ax=axes[1], shrink=0.85, label="intrusion [mm]")
fig.colorbar(c2, ax=axes[2], shrink=0.85, label="abs error [mm]")
plt.tight_layout()
plt.show()


### Simple sensitivity view

Before optimization, engineers usually want a first answer to a simpler question: which variables matter most near the current design?
The two plots below show local sensitivities at the nominal point and one-at-a-time response curves.


In [ ]:
base = x_nominal.copy()
base_y = mechanical_response(base)[0, 0]
local_slopes = []
normalized_sensitivities = []

for i in range(3):
    dx = 0.02 * (xlimits[i, 1] - xlimits[i, 0])
    x_plus = base.copy()
    x_minus = base.copy()
    x_plus[i] += dx
    x_minus[i] -= dx
    y_plus = mechanical_response(x_plus)[0, 0]
    y_minus = mechanical_response(x_minus)[0, 0]
    slope = (y_plus - y_minus) / (2.0 * dx)
    local_slopes.append(slope)
    normalized_sensitivities.append(slope * base[i] / base_y)

fig = plt.figure(figsize=(15, 8))
grid = fig.add_gridspec(2, 3, height_ratios=[1.0, 1.3])

ax_bar = fig.add_subplot(grid[0, :])
ax_bar.bar(["t", "sigma_y", "r"], normalized_sensitivities, color=["tab:blue", "tab:orange", "tab:green"])
ax_bar.axhline(0.0, color="black", linewidth=1)
ax_bar.set_ylabel("normalized local sensitivity")
ax_bar.set_title("Normalized local sensitivity at nominal design")
ax_bar.grid(axis="y", alpha=0.3)

curve_labels = ["t", "sigma_y", "r"]
curve_colors = ["tab:blue", "tab:orange", "tab:green"]

for i, (label, color) in enumerate(zip(curve_labels, curve_colors)):
    ax = fig.add_subplot(grid[1, i])
    x_line = np.tile(base, (100, 1))
    x_line[:, i] = np.linspace(xlimits[i, 0], xlimits[i, 1], 100)
    y_line = mechanical_response(x_line)
    ax.plot(x_line[:, i], y_line[:, 0], color=color, linewidth=2)
    ax.axhline(base_y, color="gray", linestyle="--", linewidth=1)
    ax.axvline(base[i], color="black", linestyle=":", linewidth=1)
    ax.set_title(f"One-at-a-time curve: {label}")
    ax.set_xlabel(var_names[i])
    ax.set_ylabel("intrusion [mm]")
    ax.grid(alpha=0.3)


plt.tight_layout()
plt.show()


## Step 3: Use the surrogate for a fast design sweep

This is where the efficiency gain appears.

Instead of running the expensive solver on thousands of concepts, we screen them with the surrogate first and keep only the most promising candidates for high-fidelity verification.


In [ ]:
n_screen = 5000
sampling_screen = LHS(xlimits=xlimits, criterion="ese", seed=101)
x_screen = sampling_screen(n_screen)

y_screen_pred = best_model.predict_values(x_screen)
rank = np.argsort(y_screen_pred[:, 0])

x_top = x_screen[rank[:10]]
y_top = y_screen_pred[rank[:10]]

print("Top 10 surrogate-ranked candidates")
print("t [mm]   sigma_y [MPa]   r [mm]   predicted intrusion [mm]")
for x_i, y_i in zip(x_top, y_top):
    print(f"{x_i[0]:6.3f}   {x_i[1]:14.1f}   {x_i[2]:6.3f}   {y_i[0]:24.3f}")


In [ ]:
# Compare surrogate predictions with the true synthetic response
# on the top-ranked candidates.

y_top_true = mechanical_response(x_top)

print("Verification of top candidates")
print("predicted   true   abs error")
for yp, yt_i in zip(y_top[:, 0], y_top_true[:, 0]):
    print(f"{yp:9.3f} {yt_i:7.3f} {abs(yp - yt_i):10.3f}")

print(f"RMSE on the top-10 verification set: {compute_rmse(best_model, x_top, y_top_true):.3f}")


## Engineering interpretation

A practical workflow for the team is:
1. run about 20 to 50 well-distributed simulations,
2. train 2 or 3 candidate surrogates,
3. validate them on a separate sample,
4. if the error is acceptable, use the surrogate for screening or pre-optimization,
5. send only the shortlisted designs back to the FE solver.

That changes the question from "Can we afford to test 5000 concepts?" to "Can we afford 30 high-quality simulations and 10 verification runs?"


## Exercise for the team

Try the following modifications:

1. Increase `ndoe` from 30 to 60. Which model benefits the most?
2. Remove one input variable and see whether `QP` becomes competitive with `KRG`.
3. Add random noise to `yt` to mimic noisy simulations. Does the preferred model change?
4. Add a second output, for example `mass`, and discuss whether the same DOE can support both outputs.
5. Replace `mechanical_response()` with data from your own simulation campaign.


## Recommendation for your real internal tutorial

For a team of simulation engineers, keep the session focused on four messages:
- surrogate modelling is a workflow, not a math lecture,
- DOE quality matters as much as model choice,
- validation is mandatory,
- surrogates are best used for screening and acceleration, not blind replacement of the solver.

If you want, the next step can be to create a second notebook based on one of your real simulation datasets instead of this synthetic example.
